# Real-Dataset Row-ID Prefilter Adapter (Colab)

This notebook demonstrates a safe adapter pattern for applying `mod30`, `mod210`, and `mod2310` wheel filters to real-dataset-style workflows.

The adapter filters **row/sample IDs** before a downstream QOS or ML workflow:

```text
X, y dataset
    → row IDs
    → wheel prefilter
    → X_filtered, y_filtered
    → downstream QOS / benchmark script
```

Guardrail: this notebook measures retained sample counts and simple baseline behavior. It does **not** claim filtering row IDs improves ML accuracy.

In [ ]:
# Clone your fork and move into repo root
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30

In [ ]:
from pathlib import Path

from modwheel import STANDARD_WHEELS
from pre_oracle_filter import pre_oracle_candidates_by_name

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

## 1. Create a real-dataset-shaped synthetic matrix

This acts as a dependency-light stand-in for a `real_datasets/` matrix such as text features, sparse features, or biological count features. The point is the adapter pattern, not the dataset itself.

In [ ]:
RANDOM_STATE = 9423
N_SAMPLES = 6000
N_FEATURES = 80

X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=20,
    n_redundant=10,
    n_classes=2,
    weights=[0.52, 0.48],
    flip_y=0.03,
    random_state=RANDOM_STATE,
)

X.shape, y.shape, y.mean()

## 2. Define row-ID wheel adapter

This is the minimal safe integration pattern for `real_datasets/` scripts: filter row IDs, then slice `X` and `y`.

In [ ]:
def filter_dataset_by_row_ids(X, y, wheel_name="mod30"):
    """Filter dataset rows using wheel-admissible sample indices."""
    row_ids = np.arange(len(X))
    filtered_ids = np.array(pre_oracle_candidates_by_name(row_ids, wheel_name), dtype=int)
    return X[filtered_ids], y[filtered_ids], filtered_ids


def class_balance(y_values):
    counts = pd.Series(y_values).value_counts().sort_index()
    return {f"class_{k}": int(v) for k, v in counts.items()}


filter_dataset_by_row_ids(X, y, "mod30")[0].shape

## 3. Retained sample table

This is the main adapter result: how many samples remain after each wheel is applied to row/sample IDs.

In [ ]:
rows = []
for wheel_name, wheel in STANDARD_WHEELS.items():
    Xf, yf, ids = filter_dataset_by_row_ids(X, y, wheel_name)
    row = {
        "wheel": wheel_name,
        "modulus": wheel.modulus,
        "residues": wheel.residue_count,
        "n_original": len(X),
        "n_retained": len(Xf),
        "retained_fraction": len(Xf) / len(X),
        "reduction_fraction": 1 - len(Xf) / len(X),
        "positive_rate_original": float(np.mean(y)),
        "positive_rate_filtered": float(np.mean(yf)),
    }
    rows.append(row)

retained = pd.DataFrame(rows)
retained

## 4. Baseline classifier sanity check

This checks whether the filtered subsets remain usable as ordinary ML inputs. This is a sanity check only, not an accuracy improvement claim.

In [ ]:
def evaluate_subset(X_subset, y_subset, label):
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset,
        y_subset,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y_subset,
    )
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    return {
        "subset": label,
        "n_samples": len(X_subset),
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    }


eval_rows = [evaluate_subset(X, y, "baseline_all_rows")]
for wheel_name in STANDARD_WHEELS:
    Xf, yf, ids = filter_dataset_by_row_ids(X, y, wheel_name)
    eval_rows.append(evaluate_subset(Xf, yf, wheel_name))

eval_df = pd.DataFrame(eval_rows)
eval_df

## 5. Save tables and figures

These outputs are ready for README, paper draft, or follow-up notebook use.

In [ ]:
fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

retained_csv = data_dir / "row_id_prefilter_retained_samples.csv"
eval_csv = data_dir / "row_id_prefilter_classifier_sanity.csv"
retained.to_csv(retained_csv, index=False)
eval_df.to_csv(eval_csv, index=False)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.bar(retained["wheel"], retained["retained_fraction"])
ax.set_title("Row-ID Prefilter Adapter: Retained Sample Fraction")
ax.set_xlabel("Wheel filter")
ax.set_ylabel("Retained sample fraction")
ax.set_ylim(0, 0.32)
ax.grid(True, axis="y", alpha=0.35)
for i, value in enumerate(retained["retained_fraction"]):
    ax.text(i, value + 0.01, f"{value:.1%}", ha="center")
fig.tight_layout()
retained_svg = fig_dir / "row_id_prefilter_retained_fraction.svg"
retained_png = fig_dir / "row_id_prefilter_retained_fraction.png"
fig.savefig(retained_svg)
fig.savefig(retained_png, dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 4.8))
plot_eval = eval_df.copy()
ax.plot(plot_eval["subset"], plot_eval["balanced_accuracy"], marker="o")
ax.set_title("Classifier Sanity Check on Filtered Row Subsets")
ax.set_xlabel("Subset")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
plt.xticks(rotation=20)
fig.tight_layout()
acc_svg = fig_dir / "row_id_prefilter_classifier_sanity.svg"
acc_png = fig_dir / "row_id_prefilter_classifier_sanity.png"
fig.savefig(acc_svg)
fig.savefig(acc_png, dpi=200)
plt.show()

retained_csv, eval_csv, retained_svg, acc_svg

## 6. Adapter path for `real_datasets/20news_svm.py`

A future non-invasive integration can wrap a real dataset script after `X, y` are loaded:

```python
from pre_oracle_filter import pre_oracle_candidates_by_name

row_ids = range(len(X))
filtered_ids = pre_oracle_candidates_by_name(row_ids, "mod30")
X = X[filtered_ids]
y = y[filtered_ids]
```

This keeps the wheel layer external to QOS internals and makes the preprocessing step easy to enable/disable.

## 7. Interpretation

- Row-ID filtering reproduces the wheel density behavior on real-dataset-shaped matrices.
- The adapter is safe because it slices `X, y` before downstream code runs.
- This notebook does not claim accuracy gains.
- Next step: adapt a specific `real_datasets/20news_*` workflow and compare runtime/sample counts.

In [ ]:
# Optional: download outputs in Colab
from google.colab import files

for path in [
    "data/row_id_prefilter_retained_samples.csv",
    "data/row_id_prefilter_classifier_sanity.csv",
    "figures/row_id_prefilter_retained_fraction.svg",
    "figures/row_id_prefilter_classifier_sanity.svg",
]:
    files.download(path)